#[2] 랭그래프

In [ ]:
!pip install -Uq langgraph langchain

In [ ]:
# langchain-google-genai 패키지 설치   langchain-google-genai
!pip install -Uq langchain-google-genai

In [ ]:
from dotenv import load_dotenv
import os
load_dotenv(dotenv_path = "/content/.env", override=True)

print(".env 내 OPENAI_API_KEY가 환경변수에 할당됐습니다:", os.environ["OPENAI_API_KEY"][:5]+"*****")
print(".env 내 GOOGLE_API_KEY가 환경변수에 할당됐습니다:", os.environ["GOOGLE_API_KEY"][:5]+"*****")

#1 . Agent 구축 기초
https://docs.langchain.com/oss/python/langgraph/quickstart

##1-1. Google Gemini 모델 사용

https://docs.langchain.com/oss/python/integrations/providers/google

###① Model 및 Tool 정의

In [ ]:
# 모델 객체 생성
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(
    model="gemini-3-flash-preview",
    # temperature=1.0,  # Gemini 3.0+ defaults to 1.0
    # max_tokens=None,
    # timeout=None,
    # max_retries=2,
    # other params...
)

In [ ]:
from langchain.tools import tool

@tool
def multiply(a: int, b: int) -> int:
    """Multiply `a` and `b`.

    Args:
        a: First int
        b: Second int
    """
    return a * b


@tool
def add(a: int, b: int) -> int:
    """Adds `a` and `b`.

    Args:
        a: First int
        b: Second int
    """
    return a + b


@tool
def divide(a: int, b: int) -> float:
    """Divide `a` and `b`.

    Args:
        a: First int
        b: Second int
    """
    return a / b

In [ ]:
# tools 정의
tools = [multiply, add, divide]

In [ ]:
# model - tool 바인딩
model_with_tools = model.bind_tools(tools)

###② State 정의

https://docs.langchain.com/oss/python/langgraph/quickstart#2-define-state

In [ ]:
# State 정의
from langchain.messages import AnyMessage
from typing_extensions import TypedDict, Annotated
from langgraph.graph.message import add_messages

class MessagesState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]
    llm_calls: int

###③ Node 정의

In [ ]:
from langchain.messages import SystemMessage

# 모델 노드 정의
def llm_call(state: MessagesState):
    sys_msg = SystemMessage(content="당신은 사칙연산을 하는 유능한 Agent입니다.")
    response = model_with_tools.invoke([sys_msg] + state["messages"])
    return {
        "messages": [response],
        "llm_calls": state.get("llm_calls", 0) + 1
    }

In [ ]:
# tool 목록 생성
tool_by_name = {tool.name: tool for tool in tools}
tool_by_name

In [ ]:
# tool_node
from langchain.messages import ToolMessage

def tool_node(state: MessagesState):
    result = []
    for tool_call in state['messages'][-1].tool_calls:
        tool = tool_by_name[tool_call['name']]
        tool_result = tool.invoke(tool_call['args'])
        result.append(ToolMessage(content=tool_result, tool_call_id=tool_call['id']))
    return {'messages' : result}


In [ ]:
# {'messages': [
#     HumanMessage(content='42 + 3 * 23은 뭔가요?', additional_kwargs={}, response_metadata={}, id='d668f973-433b-43a6-832c-12f69d7a1d47'),
#     AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 666, 'prompt_tokens': 271, 'total_tokens': 937, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 640, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-CYN0BtgC1vSXVOqkym1wLgdCHV0mg', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--e3ff5826-b8ae-45ef-b321-845634280cac-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 23}, 'id': 'call_6JIP9XWGEL4Y8gsDf7kLMCns', 'type': 'tool_call'}], usage_metadata={'input_tokens': 271, 'output_tokens': 666, 'total_tokens': 937, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 640}}),
#     ToolMessage(content='69', name='multiply', id='ce3260fd-d978-4f22-9ac8-7ecba8497b5c', tool_call_id='call_6JIP9XWGEL4Y8gsDf7kLMCns'),
#     AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 20, 'prompt_tokens': 302, 'total_tokens': 322, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-CYN0IzrIXR33PjGJkChQkJNBWEPNp', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--79f83b57-4a2b-4ceb-8dd1-79bb8ac2907c-0', tool_calls=[{'name': 'add', 'args': {'a': 42, 'b': 69}, 'id': 'call_dYzyonAijap4wIkosXukpffe', 'type': 'tool_call'}], usage_metadata={'input_tokens': 302, 'output_tokens': 20, 'total_tokens': 322, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}),
#     ToolMessage(content='111', name='add', id='a30c98b9-41e5-4496-aed3-044c02028346', tool_call_id='call_dYzyonAijap4wIkosXukpffe'),
#     AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 666, 'prompt_tokens': 333, 'total_tokens': 999, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 640, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-CYN0KeTPVjeebFxGf8IXHbmsLUKQ9', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--3e09231b-dfd0-4190-9c22-024988cc4389-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 23}, 'id': 'call_7i9W8KYCfoyL0V5YM3hA9CNM', 'type': 'tool_call'}], usage_metadata={'input_tokens': 333, 'output_tokens': 666, 'total_tokens': 999, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 640}}),
#     ToolMessage(content='69', name='multiply', id='f041ffd3-626e-4541-9ea7-db7c200a49ef', tool_call_id='call_7i9W8KYCfoyL0V5YM3hA9CNM'),
#     AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 922, 'prompt_tokens': 364, 'total_tokens': 1286, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 896, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-CYN0QZAwvbco2nLwS2UlXoKoQJ2zB', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--eb606c0c-8c16-4789-928d-2ed8e902074f-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 23}, 'id': 'call_bUU09sNKZb2CMKxPwn3dc7pW', 'type': 'tool_call'}], usage_metadata={'input_tokens': 364, 'output_tokens': 922, 'total_tokens': 1286, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 896}}),
#     ToolMessage(content='69', name='multiply', id='9ae49215-1418-4f1b-88e3-c32caa9dd511', tool_call_id='call_bUU09sNKZb2CMKxPwn3dc7pW'),
#     AIMessage(content='정답은 111입니다.\n\n설명:\n- 먼저 곱셈의 순서를 따릅니다: 3 * 23 = 69\n- 그리고 남은 더하기를 수행합니다: 42 + 69 = 111\n\n추가로, 계산 과정을 한 번에 확인하는 방법도 보여드리면:\n- 42 + 3 * 23 = 42 + (3 * 23) = 42 + 69 = 111\n\n필요한 다른 연산이 있으면 말씀해 주세요.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 115, 'prompt_tokens': 395, 'total_tokens': 510, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-CYN0Ywh5LSATE3IezdrYTcoik9Abu', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--79aa9996-99f3-4463-9986-b33c4f53d706-0', usage_metadata={'input_tokens': 395, 'output_tokens': 115, 'total_tokens': 510, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})
#     ]
# }

###④ 그래프 생성

In [ ]:
# 그래프 생성
from langgraph.graph import StateGraph, START, END

agent_builder = StateGraph(MessagesState)

agent_builder.add_node("llm_call", llm_call)
agent_builder.add_node("tool_node", tool_node)

agent_builder.add_edge(START, "llm_call")
agent_builder.add_edge("tool_node", "llm_call")


In [ ]:
# 조건부 함수 작성
def should_continue(state: MessagesState):
    last_message = state['messages'][-1]

    if last_message.tool_calls:
        return "tool_node"
    else:
        return "end"

In [ ]:
# 조건부 엣지 연결
# 주의: 이미 추가된 경우 중복 실행 시 ValueError가 발생할 수 있습니다.
agent_builder.add_conditional_edges(
    "llm_call",
    should_continue,
    {
        "tool_node": "tool_node",
        "end": END,
    },
)


In [ ]:
# 컴파일 (최종 에이전트 생성)
agent = agent_builder.compile()
agent

###⑤ 실행

In [ ]:
from langchain.messages import HumanMessage

messages = [HumanMessage(content="3과 4를 더한 후 7을 곱해줘")]
response = agent.invoke({"messages": messages})

In [ ]:
response

In [ ]:
response["messages"][-1].content[-1]['text']

##1-2. OpenAI 모델 사용

In [ ]:
!pip install -qU langchain-openai

###① Model 및 Tool 정의

In [ ]:
from langchain.chat_models import init_chat_model

model = init_chat_model("gpt-5-nano")

In [ ]:
model_with_tools = model.bind_tools(tools)

###② State 정의

https://docs.langchain.com/oss/python/langgraph/quickstart#2-define-state

In [ ]:
# State 정의
from langchain.messages import AnyMessage
from typing_extensions import TypedDict, Annotated
from langgraph.graph.message import add_messages

class MessagesState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]
    llm_calls: int

###③ Node 정의

In [ ]:
from langchain.messages import SystemMessage

def llm_call(state: MessagesState):
    """LLM이 현재 상태를 보고 답변하거나, 도구 사용을 요청하는 Node"""

    # 시스템 메시지를 맨 앞에 추가하고 기존 대화 기록(state["messages"])을 뒤에 붙여서 LLM에게 전송
    response = model_with_tools.invoke(
        [SystemMessage(content="당신은 사칙연산을 하는 유능한 Agent입니다.")] + state["messages"]
    )

    # 변경된 상태를 반환
    return {
        "messages": [response],
        "llm_calls": state.get('llm_calls', 0) + 1
    }

In [ ]:
tools_by_name = {tool.name: tool for tool in tools}

In [ ]:
from langchain.messages import ToolMessage

def tool_node(state: MessagesState):
    """LLM이 도구 사용을 요청했을 때, 실제로 도구를 실행하는 단계"""

    result = []
    # 가장 최근 메시지(LLM의 응답)에서 도구 호출 요청(tool_calls)들을 꺼내기
    for tool_call in state["messages"][-1].tool_calls:
        # 1. 도구 이름으로 실제 함수 찾기
        tool = tools_by_name[tool_call["name"]]

        # 2. 함수를 실행하여 결과를 얻기
        tool_result = tool.invoke(tool_call["args"])

        # 3. 결과를 ToolMessage 형태로 포장 (tool_call_id는 필수!)
        result.append(ToolMessage(content=tool_result, tool_call_id=tool_call["id"]))

    # 실행 결과를 대화 기록에 추가
    return {"messages": result}

###④ 그래프 생성

In [ ]:
from langgraph.graph import StateGraph, START

# 1. 워크플로우(그래프) 생성
agent_builder = StateGraph(MessagesState)

# 2. 노드(작업자) 배치
agent_builder.add_node("llm_call", llm_call)
agent_builder.add_node("tool_node", tool_node)

# 3. 엣지(연결선) 연결
agent_builder.add_edge(START, "llm_call")
agent_builder.add_edge("tool_node", "llm_call")

In [ ]:
from langgraph.graph import StateGraph, START, END

def should_continue(state: MessagesState):
    """
    LLM의 응답을 보고 다음 단계로 어디를 갈지 결정
    """
    messages = state["messages"]
    last_message = messages[-1]

    # LLM이 도구 호출(tool_calls)을 포함한 응답을 보낸 경우
    if last_message.tool_calls:
        return "tool_node"

    # 도구 호출이 없으면 작업 종료
    return END

In [ ]:
# 조건부 엣지 연결
agent_builder.add_conditional_edges(
    "llm_call",         # 출발지
    should_continue,    # 판단 로직 함수
    ["tool_node", END]  # 갈 수 있는 목적지들
)

In [ ]:
# 컴파일 (최종 에이전트 생성)
agent = agent_builder.compile()

In [ ]:
agent

###⑤ 실행

In [ ]:
from langchain.messages import HumanMessage

messages = [HumanMessage(content="3과 4를 더하고, 7을 곱해줘")]
response = agent.invoke({"messages": messages})

In [ ]:
response

In [ ]:
response["messages"][-1].content

#2 . Agent 구축 실습
https://docs.langchain.com/oss/python/langgraph/thinking-in-langgraph

##2-1. 실습 1

###① Model 및 Tool 정의

In [ ]:
from langchain.chat_models import init_chat_model

model = init_chat_model("gpt-5-nano")

###② State 정의

In [ ]:
from typing_extensions import TypedDict

# 단순 자료형은 값이 들어올 때마다 덮어쓰기(Override) 됩니다.
class AgentState(TypedDict):
    email_content: str   # 고객이 보낸 원문 이메일
    category: str        # inquiry(문의) / complaint(불만)
    response: str        # 중간/최종 결과물

###③ Node 정의

In [ ]:
# [Node 1] 이메일 읽기
def read_email(state: AgentState):
    return {"email_content": state["email_content"]}

In [ ]:
# [Node 2] 의도 분류 (Router 로직의 기준점)
def classify_intent(state: AgentState):
    print("\n[2] 이메일 내용을 분석합니다...")
    # 앞선 노드가 적어둔 이메일 내용을 State에서 읽어옵니다.
    email_cont = state["email_content"]

    # 간단한 키워드 기반의 하드코딩 규칙 분류
    if "환불" in email_cont or "빨리" in email_cont or "불만" in email_cont:
        category = "complaint"   # 불만 -> 상담원 연결
    else:
        category = "inquiry"   # 문의 -> 매뉴얼 검색

    # 반환하는 즉시 State의 category에 새 값으로 덮어씌워집니다.
    return {"category": category}

In [ ]:
# [Node 3-A] 상담원 이관 (긴급 이슈 트랙)
def escalate_to_human(state : AgentState):
    print("3-A 진입... 상담원 이관합니다.")
    return {"response" : "상담원이 곧 연락드릴 예정입니다. 불편을 드려 죄송합니다"}

# [Node 3-B] 매뉴얼 검색 (AI 자동 처리 트랙)
def search_manual(state : AgentState):
    print("3-B 진입... 메뉴얼을 검색합니다")
    return

In [ ]:
# [Node 4] 답변 생성
def write_reply(state: AgentState):
    email_cont = state["email_content"]
    response = model.invoke(f"이메일 내용: {email_cont}")
    return {"response": response}

###④ 그래프 생성

In [ ]:
from langgraph.graph import StateGraph, START, END

# 1. 워크플로우(그래프) 생성
agent_builder = StateGraph(AgentState)

# 2. 노드(작업자) 배치
agent_builder.add_node("read_email", read_email)
agent_builder.add_node("classify_intent", classify_intent)
agent_builder.add_node("search_manual", search_manual)
agent_builder.add_node("escalate_to_human", escalate_to_human)
agent_builder.add_node("write_reply", write_reply)

# 3. 엣지(연결선) 연결
agent_builder.add_edge(START, "read_email")
agent_builder.add_edge("read_email", "classify_intent")


In [ ]:
# 라우팅 함수: State의 정보를 보고 어디로 갈지 문자열로 반환
def route_email(state: AgentState):
  category = state["category"]
  if category == "complaint":
      return "escalate_to_human"
  else :
      return "search_manual"

In [ ]:
# 4. 조건부 엣지 연결
agent_builder.add_conditional_edges(
    "classify_intent",
    route_email,
    ["escalate_to_human", "search_manual"],
)

In [ ]:
# 5. 나머지 엣지(연결선) 연결
agent_builder.add_edge("search_manual", "write_reply")
agent_builder.add_edge("write_reply", END)
# agent_builder.add_edge("escalate_to_human", END)
# escalate_to_human은 목적지가 없으므로, 따로 엣지를 잇지 않아도 암묵적으로 END로 향합니다.

In [ ]:
# 6. 컴파일 (최종 에이전트 생성)
agent = agent_builder.compile()

In [ ]:
agent

###⑤ 실행

In [ ]:
inputs = {"email_content": "비밀번호 변경 방법을 알려주세요."}
response = agent.invoke(inputs)
print(response['response'])

In [ ]:
response

In [ ]:
inputs = {"email_content": "당장 환불해줘!"}
response = agent.invoke(inputs)

In [ ]:
response

In [ ]:
inputs = {"email_content": "당장 환급해줘!"}
response = agent.invoke(inputs)

In [ ]:
response

##2-2. 실습 2

###① Model 및 Tool 정의

In [ ]:
from langchain.chat_models import init_chat_model

model = init_chat_model("gpt-5-nano")

In [ ]:
from langchain.tools import tool

@tool
def search_manual(query : str) -> str :

    """
    고객의 질문에 답변하기 위해 참고할만한 규정이나 매뉴얼을 검색할때 사용하는 도구
    """
    if '비밀번호 변경' in query :
        return '비밀번호 변경은 MyPage의 보안 설정에서 할수 있습니다'
    elif '배송' in query :
        return 'oo 택배로 3일 이내 배송 예정입니다'
    else :
        return '해당 내용 관련 매뉴얼은 찾을 수 없습니다'


In [ ]:
tools = [search_manual]
# tools

In [ ]:
model_with_tools = model.bind_tools(tools)

In [ ]:
model_with_tools

###② State 정의

In [ ]:
from langchain.messages import AnyMessage
from typing_extensions import TypedDict, Annotated
from langgraph.graph.message import add_messages

class AgentState(TypedDict):
    messages : Annotated[list[AnyMessage], add_messages]
    category : str

###③ Node 정의

In [ ]:
# [Node 1] 분류 노드 (Manager)
# LLM에게 이메일을 읽고 어떤 트랙으로 보낼지 결정
from langchain.messages import SystemMessage

def classify_node(state: AgentState):
    print("\n--- [1] 분류 단계 (LLM 판단) ---")
    last_message = state["messages"][-1]

    # 분류를 위한 프롬프트 (System Prompt)
    prompt = """
    당신은 고객 센터 관리자입니다. 고객의 이메일을 분석해서 다음 단계를 결정하세요.

    1. 단순 문의나 정보 요청이라면 -> 'inquiry' 반환
    2. 환불 요청, 불만 제기, 화난 고객이라면 -> 'complaint' 반환

    답변은 오직 단어 하나만 하세요.
    """

    # LLM 호출
    response = model.invoke([SystemMessage(content=prompt), last_message])
    raw_content = response.content

    if isinstance(raw_content, list):
        # Gemini 등: 리스트로 오는 경우 텍스트 블록만 추출해서 합침
        decision = "".join([block['text'] for block in raw_content if block.get('type') == 'text'])
    else:
        # OpenAI 등: 문자열로 오는 경우 그대로 사용
        decision = str(raw_content)

    # 공백 제거 및 소문자 변환
    decision = decision.strip().lower()
    print(f"   -> LLM 판단 결과: {decision}")

    # 분류 결과를 를 상태에 저장
    if "inquiry" in decision:
        return {"category": "inquiry"}
    else:
        return {"category": "complaint"}

In [ ]:
# [Node 2] 상담 AI 노드 (Worker)
# 도구를 사용해서 답변을 생성하는 실제 에이전트
def consultant_node(state: AgentState):
    print("\n--- [2-A] 상담 AI 답변 생성 중 ---")
    response = model_with_tools.invoke(state["messages"])
    return {"messages": [response]}


In [ ]:
from langchain_core.messages import AIMessage

# [Node 3] 상담원 이관 노드
def escalate_node(state: AgentState):
    print("\n--- [3-A] 상담원 이관 ---")
    return {"messages" : [AIMessage(content="해당 메일은 상담원에게 이관되었습니다")]}


In [ ]:
tools = [search_manual]

In [ ]:
tools

In [ ]:
tools_by_name = {tool.name: tool for tool in tools}

In [ ]:
tools_by_name

In [ ]:
from langchain.messages import ToolMessage
# [Node 4] Tool Node
def tool_node(state: AgentState):
    print("\n--- [Tool Node] 도구 직접 실행 ---")
    result = []
    # 가장 최근 메시지에서 도구 호출 요청 꺼내기
    last_message = state["messages"][-1]

    for tool_call in last_message.tool_calls:
        # 1. 도구 이름으로 실제 함수 찾기
        tool = tools_by_name[tool_call["name"]]

        # 2. 함수를 실행하여 결과를 얻기
        print(f"   -> 실행 중: {tool_call['name']}")
        tool_result = tool.invoke(tool_call["args"])

        # 3. 결과를 ToolMessage 형태로 포장 (tool_call_id 필수!)
        result.append(ToolMessage(content=str(tool_result), tool_call_id=tool_call["id"]))

    # 실행 결과를 대화 기록에 추가
    return {"messages": result}

###④ 그래프 생성

In [ ]:
from langgraph.graph import StateGraph, START, END

# 1. 워크플로우(그래프) 생성
agent_builder = StateGraph(AgentState)

# 2. 노드(작업자) 배치
agent_builder.add_node("classify_node", classify_node)
agent_builder.add_node("consultant_node", consultant_node)
agent_builder.add_node("escalate_node", escalate_node)
agent_builder.add_node("tool_node", tool_node)

# 3. 엣지(연결선) 연결
agent_builder.add_edge(START, "classify_node")


In [ ]:
# 4. 조건부 엣지 연결
# 라우터 1: 분류 결과에 따른 분기
def route_after_classify(state: AgentState):
    category = state["category"]
    if category == "inquiry":
        return "consultant"
    else:
        return "escalate"

In [ ]:
# 라우터 1: 조건부 엣지 연결
agent_builder.add_conditional_edges(
    "classify_node",
    route_after_classify,
    {
        "consultant" : "consultant_node",
        "escalate" : "escalate_node"
    },
)

In [ ]:
# 라우터 2: 도구 사용 여부 판단
def should_continue(state: AgentState):
    last_message = state["messages"][-1]

    # LLM이 도구 호출(tool_calls)을 포함한 응답을 보낸 경우
    if last_message.tool_calls:
        return "tool_node"

    # 도구 호출이 없으면 종료
    return END

In [ ]:
# 라우터 2: 조건부 엣지 연결 - 상담 AI 후 도구 사용 여부 판단
agent_builder.add_conditional_edges(
    "consultant_node",
    should_continue,
    ["tool_node", END]
)

In [ ]:
# 5. 나머지 엣지(연결선) 연결
agent_builder.add_edge("tool_node", "consultant_node")
agent_builder.add_edge("escalate_node", END)

In [ ]:
# 6. 컴파일 (최종 에이전트 생성)
agent = agent_builder.compile()

In [ ]:
agent

###⑤ 실행

In [ ]:
from langchain.messages import HumanMessage, SystemMessage

inputs = {"messages": [HumanMessage(content="비밀번호 변경은 어디서 해?")]}
response = agent.invoke(inputs)

In [ ]:
response

In [ ]:
response["messages"][-1].content

In [ ]:
inputs = {"messages": [HumanMessage(content="당장 환불해줘!")]}
response = agent.invoke(inputs)

In [ ]:
response

In [ ]:
response["messages"][-1].content

State 설계 원칙
([공식 문서](https://docs.langchain.com/oss/python/langgraph/thinking-in-langgraph#what-belongs-in-state))